In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [101]:
key = pd.read_csv('https://raw.githubusercontent.com/HSEAi2025Team-7/keyRateForecast/Alex/cbr.csv')

In [102]:
key.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   date                100 non-null    object 
 1   change_key_rate     96 non-null     object 
 2   key_rate_last_diff  100 non-null    float64
 3   key_rate            95 non-null     float64
 4   inflation           50 non-null     float64
dtypes: float64(3), object(2)
memory usage: 4.0+ KB


In [114]:
# заполняем медианой change_key_rate
key['change_key_rate'] = key['change_key_rate'].fillna('сохранить')
# заполняем медианой key_rate
med_key_rate = key['key_rate'].median()
key['key_rate'] = key['key_rate'].fillna(med_key_rate)
# заполняем пропуск медианой
med_inflation = key["inflation"].median()
key["inflation"] = key["inflation"].fillna(med_inflation)
# добавляем колонку цель по инфляции = 4
key['inflation_target'] = 4
# добавляем колонку разница между инфляцией и целью по инфляции
key['difference'] = key['inflation'] - key['inflation_target']
# добавляем колонку разница межды инфляцией
key['inflation_diff'] = key['inflation'].diff().shift(-1)
# заполняем пропуск медианой
med_inflation_diff = key["inflation_diff"].median()
key["inflation_diff"] = key["inflation_diff"].fillna(med_inflation_diff)
# прошлая ключевая ставка
key['key_rate_last'] = key['key_rate'].shift(-1)
med_key_rate_last = key['key_rate_last'].median()
key['key_rate_last'] = key['key_rate_last'].fillna(med_key_rate_last)

In [115]:
key.isna().sum()

,0
date,0
change_key_rate,0
key_rate_last_diff,0
key_rate,0
inflation,0
inflation_target,0
difference,0
inflation_diff,0
key_rate_last,0


In [158]:
key.head()

,date,change_key_rate,key_rate_last_diff,key_rate,inflation,inflation_target,difference,inflation_diff,key_rate_last
0,24 октября 2025,снизить,50.0,16.5,8.2,4,4.2,0.0,17.0
1,12 сентября 2025,снизить,100.0,17.0,8.2,4,4.2,1.0,18.0
2,25 июля 2025,снизить,200.0,18.0,9.2,4,5.2,0.6,20.0
3,6 июня 2025,снизить,100.0,20.0,9.8,4,5.8,0.5,21.0
4,25 апреля 2025,сохранить,0.0,21.0,10.3,4,6.3,-0.1,21.0


In [118]:
# делим на матрицу признаков и целевую переменную
X1 = key[['inflation', 'inflation_target', 'difference', 'inflation_diff', 'key_rate_last', 'key_rate_last_diff']]
y1 = key['change_key_rate']

In [148]:
# делим на матрицу признаков и целевую переменную
X2 = key[['inflation', 'inflation_target', 'difference', 'inflation_diff', 'key_rate_last', 'key_rate_last_diff']]
y2 = key['key_rate']

In [57]:
X1.head()

,inflation,inflation_target,difference,inflation_diff,key_rate_last,key_rate_last_diff
0,8.2,4,4.2,0.0,17.0,50.0
1,8.2,4,4.2,1.0,18.0,100.0
2,9.2,4,5.2,0.6,20.0,200.0
3,9.8,4,5.8,0.5,21.0,100.0
4,10.3,4,6.3,-0.1,21.0,0.0


In [58]:
y1.head()

,change_key_rate
0,снизить
1,снизить
2,снизить
3,снизить
4,сохранить


Рабочая модель

In [154]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, mean_absolute_error, mean_squared_error, r2_score, accuracy_score

# Разделение данных
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.20, random_state=42 , stratify=y1
)

# Масштабирование
scaler = StandardScaler()
X1_train_scaled = scaler.fit_transform(X1_train)
X1_test_scaled = scaler.transform(X1_test)

# Создание модели с фиксированными параметрами
model = LogisticRegression(
    class_weight='balanced',
    penalty= 'l2',
    solver= 'liblinear', #'saga'
)

# Обучение модели
model.fit(X1_train_scaled, y1_train)

# Оценка модели на тесте

y1_pred = model.predict(X1_test_scaled)

In [155]:
print("Accuracy:", accuracy_score(y1_test, y1_pred))
print("f1:", f1_score(y1_test, y1_pred, average='micro'))

Accuracy: 0.6
f1: 0.6


In [156]:
print("Отчёт классификации:\n", classification_report(y1_test, y1_pred, target_names=['Понижение ставки', 'Сохраняется','Повышение ставки']))
print("Матрица ошибок:\n", confusion_matrix(y1_test, y1_pred))

Отчёт классификации:
                   precision    recall  f1-score   support

Понижение ставки       0.50      0.50      0.50         4
     Сохраняется       0.50      0.14      0.22         7
Повышение ставки       0.64      1.00      0.78         9

        accuracy                           0.60        20
       macro avg       0.55      0.55      0.50        20
    weighted avg       0.56      0.60      0.53        20

Матрица ошибок:
 [[2 1 1]
 [2 1 4]
 [0 0 9]]


In [157]:
print(y1_pred)
print(y1_test)

['сохранить' 'сохранить' 'повысить' 'сохранить' 'повысить' 'сохранить'
 'сохранить' 'сохранить' 'сохранить' 'повысить' 'сохранить' 'сохранить'
 'повысить' 'снизить' 'снизить' 'сохранить' 'сохранить' 'сохранить'
 'сохранить' 'сохранить']
5     сохранить
43    сохранить
47      снизить
82    сохранить
53      снизить
93      снизить
73      снизить
14    сохранить
94     повысить
39     повысить
71      снизить
13    сохранить
18     повысить
67      снизить
35     повысить
7     сохранить
25    сохранить
79      снизить
83    сохранить
12    сохранить
Name: change_key_rate, dtype: object


In [149]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, mean_absolute_error, mean_squared_error, r2_score, accuracy_score

# your code here
# Разделение данных
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.20, random_state=42
    )

    # Масштабирование
scaler = StandardScaler()
X2_train_scaled = scaler.fit_transform(X2_train)
X2_test_scaled = scaler.transform(X2_test)

    # Создание модели с фиксированными параметрами
model = LinearRegression()

                # Обучение модели
model.fit(X2_train_scaled, y2_train)

                # Оценка модели на тесте

y2_pred = model.predict(X2_test_scaled)

In [150]:
mse = mean_squared_error(y2_test, y2_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y2_test, y2_pred)
mape = np.mean(np.abs((y2_test - y2_pred) / y2_test)) * 100  # в процентах
R2 = r2_score(y2_test, y2_pred)
# Вывод результатов
print(f"MSE:  {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MAE:  {mae:.6f}")
print(f"MAPE: {mape:.6f}%")
print(f"R2:   {R2:.6f}")

MSE:  5.996213
RMSE: 2.448717
MAE:  1.482609
MAPE: 13.501101%
R2:   0.714081


In [151]:
print(y2_pred)
print(y2_test)

[11.10417848  7.82906363  9.63024285  4.95563381  4.88828119  5.44354487
  6.79127002 11.10417848 15.89799312 16.89508378  9.19095884 17.4991684
 10.46067393  9.3066535  14.07613997 20.57530576 10.08073594 10.60885156
 15.97066241  9.16697199]
83     9.00
53     7.00
70     9.00
45     4.25
44     4.25
39     5.50
22     7.50
80    11.00
10    18.00
0     16.50
18    12.00
30    14.00
73     9.75
33     9.00
90     9.00
4     21.00
76    10.00
77    10.00
12    16.00
31    17.00
Name: key_rate, dtype: float64
